# Audio Extraction

In [1]:
import math
import os
from dotenv import load_dotenv
load_dotenv()
print(f'\tOPENAI_API_KEY={os.getenv("OPENAI_API_KEY")[:20]}')

	OPENAI_API_KEY=sk-proj-iKU13YeoxNgF


In [ ]:
# step1 : 동영상 -> 오디오 추출
# step2 : 오디오를 10분 단위로 분할
# step3 : 각 분할 오디오로 Whisper API 호출 -> 전사문 (transcript) 
# step4 : 모든 전사문 합치기

## ffmpeg 사용: mp4 -> mp3 음성 추출

In [ ]:
# CLI 에서 실행해보자.

# >ffmpeg -i mp4파일명 -vn mp3파일명 -y
# -i 옵션 : input. 입력 파일 지정 (여러개 지정 가능)
# -vn 옵션 : disable video. 비디오는 무시하고 오디오만 추출
# -y 옵션 : 덮어쓰기 할때 y/n 사용자 입력 대기하지 않고 yes 로 진행

# 실행결과=> autio.mp3 생성됨  <- 폴더에서 확인해보자.

In [ ]:
# >ffmpeg -i D:\MCP2607\LangWork\dataset\mp4\podcast.mp4 -vn D:\MCP2607\LangWork\dataset\mp4\audio.mp3 -y   

## 오디오 추출 함수 작성

In [2]:
import subprocess

In [3]:
base_path = r'D:\MCP2607\LangWork\dataset\mp4'  # 동영상 경로
out_path = r'D:\MCP2607\LangWork\dataset\mp4\out'  # 오디오 분할 추출 경로

import os
if not os.path.exists(out_path):
    os.makedirs(out_path)

video_file = 'podcast.mp4'
audio_file = 'audio.mp3'

src_path = os.path.join(base_path, video_file)
dst_path = os.path.join(base_path, audio_file)

print(src_path)
print(dst_path)


D:\MCP2607\LangWork\dataset\mp4\podcast.mp4
D:\MCP2607\LangWork\dataset\mp4\audio.mp3


In [4]:
def extract_audio_from_video(video_path, audio_path):
    # CLI 에서 실행할 command 를 준비
    command = ["ffmpeg", "-i", video_path, "-vn", audio_path, "-y"]
    subprocess.run(command)

In [5]:
extract_audio_from_video(src_path, dst_path)

# Cutting the Audio

## pydub 패키지

In [ ]:
# pydub 패키지
#  공식: https://github.com/jiaaro/pydub 
#  pip install pydub  <- 설치 필요  (사전에 ffmpeg 가 설치되고 경로 설정도 되어 있어야 한다)

In [6]:
from pydub import AudioSegment

In [7]:
track = AudioSegment.from_mp3(dst_path)

In [9]:
# track

In [10]:
track.duration_seconds  # 몇초 분량의 오디오

4422.426122448979

In [11]:
len(track)   # ms

4422426

In [12]:
# 오디오의 첫 5분을 선택
five_minutes = 5 * 60 * 1000   # 단위: ms
first_five = track[:five_minutes]

In [14]:
# first_five

In [15]:
first_five.duration_seconds

300.0

In [17]:
first_five.export(os.path.join(out_path, 'first_five.mp3'), format='mp3')

<_io.BufferedRandom name='D:\\MCP2607\\LangWork\\dataset\\mp4\\out\\first_five.mp3'>

## 오디오 파일 분할

In [18]:
ten_minutes = 10 * 60 * 1000

In [19]:
len(track) / ten_minutes

7.37071

In [20]:
chunks = math.ceil(len(track) / ten_minutes)
chunks  # 분할될 오디오 덩어리들의 개수

8

In [22]:
for i in range(chunks):
    start_time = i * ten_minutes
    end_time = (i + 1) * ten_minutes
    # print(f"{i} start: {start_time} end {end_time}")

    chunk = track[start_time:end_time]
    chunk.export(os.path.join(out_path, f'chunk_{i}.mp3'), format='mp3')

## 오디오 분할 함수

In [23]:
# audio_path : 원본 오디오 경로
# chunk_size : minute
# chunks_folder: chunk 들을 저장할 폴더
def cut_audio_in_chunks(audio_path, chunk_size, chunks_folder):
    track = AudioSegment.from_mp3(audio_path)
    # print(len(track))
    chunk_len = chunk_size * 60 * 1000
    chunks = math.ceil(len(track) / chunk_len)
    # print(chunks)
    for i in range(chunks):
        start_time = i * chunk_len
        end_time = (i + 1) * chunk_len

        chunk = track[start_time:end_time]

        exp_path = os.path.join(chunks_folder, f"chunk_{i}.mp3")
        chunk.export(exp_path, format="mp3")                     

In [24]:
cut_audio_in_chunks(dst_path, 10, out_path)

# Whisper Transcript

## Whisper API 다루기

In [25]:
import openai

In [26]:
transcript = openai.audio.transcriptions.create(
    model='whisper-1',
    file=open(os.path.join(out_path, 'chunk_0.mp3'), 'rb'),
    language='en',
)

transcript

Transcription(text="If success is this lagging indicator of commitment now, how can you be sure that you are paying your dues? The best-selling author and host. The number one health and wellness podcast. On Purpose with Jay Shetty. Society has gone in the direction of becoming addicted to pleasure. Yes. Or pleasure-seeking. Where, from the Stoic's perspective, why did we even ever go down that road? Like, why did we leave wisdom and self-control? Or did we never have it at all and we've always been trying to balance it? Yeah. I mean, I guess that's the big question is like, why do we take something that we like too far? Yeah. Right. So the Epicureans would say like, look, drinking is great, but if you have a hangover the next day, was it actually so great? And so, you know, if you, if you push the pleasure too far, it becomes not pleasurable, but in the moment that feels very far away, right? Like in the moment you want the thing now. Obviously sex is this thing for people. It's like 

In [27]:
print(transcript.text)

If success is this lagging indicator of commitment now, how can you be sure that you are paying your dues? The best-selling author and host. The number one health and wellness podcast. On Purpose with Jay Shetty. Society has gone in the direction of becoming addicted to pleasure. Yes. Or pleasure-seeking. Where, from the Stoic's perspective, why did we even ever go down that road? Like, why did we leave wisdom and self-control? Or did we never have it at all and we've always been trying to balance it? Yeah. I mean, I guess that's the big question is like, why do we take something that we like too far? Yeah. Right. So the Epicureans would say like, look, drinking is great, but if you have a hangover the next day, was it actually so great? And so, you know, if you, if you push the pleasure too far, it becomes not pleasurable, but in the moment that feels very far away, right? Like in the moment you want the thing now. Obviously sex is this thing for people. It's like the thing you're att

## 녹취록 생성 함수

In [30]:
import glob
import re

In [ ]:
def natural_key(path):
    # 파일명에서 숫자 부분만 추출해서 정수로 변환
    return  [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', path)]    
    


def transcribe_chunks(chunk_folder, destination):
    files = glob.glob(os.path.join(chunk_folder, "chunk*.mp3"))
    # files.sort()
    files.sort(key=natural_key)

    for file in files:
        with open(file, "rb") as audio_file, open(destination, "a") as text_file:
            print(file, '녹취록 가져오는 중...', end="")
            # 각 chunk 별로 녹취록 작성.
            transcript = openai.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file,
                language="en"
            )

            text_file.write(transcript.text)   # 텍스트 파일에 추가(append)
            print('완료')
    
# 테스트
transcribe_chunks(out_path, os.path.join(out_path, 'transcript.txt'))


D:\MCP2607\LangWork\dataset\mp4\out\chunk_0.mp3 녹취록 가져오는 중...완료
D:\MCP2607\LangWork\dataset\mp4\out\chunk_1.mp3 녹취록 가져오는 중...